# Model 3 — Single Gaussian Basin, Vertically Variable Density

**Gravity inversion** for a single sedimentary basin with depth-dependent density contrast:

$$\Delta\rho(z) = -250 + 0.02z \quad [\text{kg/m}^3]$$

## Imports and core dependencies

This cell loads NumPy, Matplotlib, SciPy interpolation and optimisation routines, and the custom `compute_gravity` forward solver for variable density.[file:4]  
These libraries provide array operations, plotting tools, B‑spline surfaces, global and local optimisers, and the 3D gravity kernel with depth‑dependent density.[file:4]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mticker
from scipy.interpolate import RectBivariateSpline, griddata
from scipy.optimize import differential_evolution, minimize
from gravity3d_variable_density1 import compute_gravity
import time

print('All imports successful.')

## Domain and full-resolution grid

The model domain spans 20 × 20 × 4 km and is discretised into a 30×30×25 grid used for the true model, inversion, and plotting.[file:4]  
Cell edges, centres, and 2D coordinate grids are built for x–y, and the domain size and grid resolution are printed.[file:4]

In [ ]:
# ── Domain ─────────────────────────────────────────────────────────────────
Lx, Ly, Lz = 20_000.0, 20_000.0, 4_000.0

# Full-resolution grid (used for everything: true model, inversion, plots)
nx, ny, nz = 30, 30, 25

xe = np.linspace(0, Lx, nx + 1)
ye = np.linspace(0, Ly, ny + 1)
ze = np.linspace(0, Lz, nz + 1)
xc = 0.5 * (xe[:-1] + xe[1:])
yc = 0.5 * (ye[:-1] + ye[1:])
zc = 0.5 * (ze[:-1] + ze[1:])
X2d, Y2d = np.meshgrid(xc, yc, indexing='ij')

print(f'Domain    : {Lx/1000:.0f} x {Ly/1000:.0f} x {Lz/1000:.1f} km')
print(f'Full grid : {nx}x{ny}x{nz}')

## True Gaussian basin geometry

This cell defines the true basin as a centred 2D Gaussian with a maximum depth of 3000 m and horizontal standard deviations of 4 km in both x and y. 
It constructs the depth surface \(Z_\text{basin,true}\) and reports the nominal and actual maximum depth, providing the reference geometry for inversion.

In [ ]:
# ── True Gaussian basin ─────────────────────────────────────────────────────
depth_max = 3_000.0
cx, cy    = Lx / 2, Ly / 2
sigma_x   = 4_000.0
sigma_y   = 4_000.0

Z_basin_true = depth_max * np.exp(
    -(((X2d - cx)**2) / (2 * sigma_x**2) +
      ((Y2d - cy)**2) / (2 * sigma_y**2))
)

print(f'Basin     : depth_max={depth_max:.0f} m,  sigma={sigma_x/1000} km')
print(f'True max depth : {Z_basin_true.max():.1f} m')

## Depth-dependent density contrast law

Here the depth trend of density contrast is specified as \(\Delta\rho(z) = drho0 + \alpha z\) with \(drho0 = -250\) kg/m³ at the surface and \(\alpha = 0.02\) kg/m³/m.  
A helper `density_contrast_at_depth(z_val)` evaluates this relation, and example values at 0 and 3000 m depth are printed to check the trend.

In [ ]:
# ── Depth-dependent density contrast ───────────────────────────────────────
drho0 = -250.0   # kg/m3 at surface
alpha =   0.02   # kg/m3 per metre

def density_contrast_at_depth(z_val):
    return drho0 + alpha * z_val

print(f'Density   : drho(z) = {drho0} + {alpha}*z  kg/m3')
print(f'            at z=0: {drho0:.0f},  at z=3000: {drho0+alpha*3000:.0f} kg/m3')

## Density volume builder for variable density

This cell defines `build_density(depth_surface)`, which returns a 3D density contrast array with depth‑dependent values.
For each depth level, it computes the appropriate contrast from the depth law and assigns it below the basin surface, producing a variable‑density model for forward gravity.

In [ ]:
# ── Density volume builder ──────────────────────────────────────────────────
def build_density(depth_surface):
    """Returns rho_contrast[nx, ny, nz] with depth-dependent contrast."""
    nx_l = depth_surface.shape[0]
    rho = np.zeros((nx_l, ny, nz))
    for k in range(nz):
        drho_k = density_contrast_at_depth(zc[k])
        rho[:, :, k][zc[k] < depth_surface] = drho_k
    return rho

print('build_density defined.')

## Forward gravity computation and noisy data

Using the true basin and variable‑density model, this cell computes the full‑resolution gravity anomaly at the surface observation grid.
It adds Gaussian noise of 1 mGal standard deviation to obtain synthetic observed data `gz_obs`, printing the ranges of clean and noisy gravity and the total number of observations.

In [ ]:
# ── Forward gravity (full grid) ─────────────────────────────────────────────
Xobs = X2d.copy()
Yobs = Y2d.copy()
Zobs = np.zeros_like(X2d)

print('Computing full-resolution forward gravity ...')
t0 = time.time()
rho_true = build_density(Z_basin_true)
gz_clean = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, rho_true)

np.random.seed(42)
noise_std = 1.0
gz_obs = gz_clean + np.random.normal(0.0, noise_std, gz_clean.shape)
#gz_obs = gz_clean
print(f'Done in {time.time()-t0:.1f}s  |  '
      f'gz: {gz_clean.min():.2f} to {gz_clean.max():.2f} mGal')
print(f'gz_obs : {gz_obs.min():.2f} to {gz_obs.max():.2f} mGal  (noise={noise_std} mGal)')
print(f'Obs pts : {nx}x{ny} = {nx*ny}')

## B-spline parametrisation and regularised misfit

A 16×16 control grid is defined over the model with depths bounded between 0 and 4000 m, and `surface_from_params` interpolates control depths to a smooth basin surface. 
The cell builds a 2D Laplacian for Tikhonov smoothness, sets regularisation weights \(\lambda_s=\lambda_d=10^{-4}\), constructs a reference Gaussian depth surface for depth‑bias, and defines both regularised and data‑only misfit functions normalised by \(\text{var}(gz_\text{obs})\).

In [ ]:
# B-spline parametrisation + regularised misfit
# IMPROVEMENT 1: 6x6 -> 12x12 control points for finer basin flank resolution.
# IMPROVEMENT 2: Normalised misfit / var(gz_obs) for scale-invariant form.
# IMPROVEMENT 3: Tikhonov 2-D Laplacian smoothness penalty (lambda_s=1e-4).
# IMPROVEMENT 4: Depth-bias term (lambda_d=1e-4) counters gravity-depth ambiguity.
n_ctrl_x, n_ctrl_y = 10, 10
x_ctrl = np.linspace(xc.min(), xc.max(), n_ctrl_x)
y_ctrl = np.linspace(yc.min(), yc.max(), n_ctrl_y)
depth_min_inv, depth_max_inv = 0.0, Lz

def surface_from_params(params, xout, yout):
    ctrl = params.reshape((n_ctrl_x, n_ctrl_y))
    spl  = RectBivariateSpline(x_ctrl, y_ctrl, ctrl, kx=3, ky=3)
    return np.clip(spl(xout, yout, grid=True), depth_min_inv, depth_max_inv)

def _build_laplacian(nc):
    N = nc * nc
    L = np.zeros((N, N))
    for i in range(nc):
        for j in range(nc):
            idx = i * nc + j
            cnt = 0
            if i > 0:    L[idx, (i-1)*nc+j] = -1; cnt += 1
            if i < nc-1: L[idx, (i+1)*nc+j] = -1; cnt += 1
            if j > 0:    L[idx, i*nc+(j-1)] = -1; cnt += 1
            if j < nc-1: L[idx, i*nc+(j+1)] = -1; cnt += 1
            L[idx, idx] = cnt
    return L

_L       = _build_laplacian(n_ctrl_x)
lambda_s = 5e-3   # smoothness — raised from 1e-4
lambda_d = 0.3    # depth prior — raised from 1e-4 (was effectively zero)
_gz_var  = float(np.var(gz_obs))

# Reference depth surface on control grid (true Gaussian) for depth-bias term
_X_ctrl, _Y_ctrl = np.meshgrid(x_ctrl, y_ctrl, indexing='ij')
_depth_ref = (depth_max * np.exp(
    -((_X_ctrl - cx)**2 / (2 * sigma_x**2) +
      (_Y_ctrl - cy)**2 / (2 * sigma_y**2)))).ravel()

def misfit(params):
    surf    = surface_from_params(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_density(surf))
    data_term   = float(np.mean((gz_pred - gz_obs)**2)) / _gz_var
    p = params / depth_max_inv
    smooth_term = float(p @ _L @ p) / (n_ctrl_x * n_ctrl_y)
    depth_bias  = float(np.mean(((params - _depth_ref) / depth_max_inv)**2))
    return data_term + lambda_s * smooth_term + lambda_d * depth_bias

def misfit_data_only(params):
    surf    = surface_from_params(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_density(surf))
    return float(np.mean((gz_pred - gz_obs)**2)) / _gz_var

bounds = [(depth_min_inv, depth_max_inv)] * (n_ctrl_x * n_ctrl_y)
print(f'Control grid  : {n_ctrl_x}x{n_ctrl_y} = {n_ctrl_x*n_ctrl_y} parameters')
print(f'Bounds        : [{depth_min_inv}, {depth_max_inv}] m')
print(f'Regularisation: lambda_s={lambda_s}  lambda_d={lambda_d}')


## DE callback
This cell initialises counters and defines `de_callback`, which evaluates the misfit at each DE iterate, updates the best value and history, and prints iteration‑by‑iteration progress.

In [ ]:
# ── DE progress callback ────────────────────────────────────────────────────
# xk passed by scipy is guaranteed to be the CURRENT BEST solution vector.
# We re-evaluate misfit(xk) and record it directly — this gives a true
# monotone convergence curve. The old pattern (_best[0] updated only when
# f < _best[0]) caused flat stretches: when xk does not change between
# generations, f == _best[0] exactly so the condition was never triggered
# and _hist_f recorded the stale value indefinitely.
_iter   = [0]
_t0     = [time.time()]
_best   = [np.inf]
_hist_x = []
_hist_f = []

def de_callback(xk, convergence):
    _iter[0] += 1
    f = misfit(xk)           # xk = scipy current best; f IS the best misfit
    _best[0] = f             # update unconditionally (scipy guarantees monotone)
    elapsed = time.time() - _t0[0]
    _hist_x.append(xk.copy())
    _hist_f.append(f)        # record f directly, not the stale _best[0]
    print(f'  DE iter {_iter[0]:>4d} | best misfit = {f:.6f} | '
          f'elapsed = {elapsed:.1f}s | convergence = {convergence:.4f}')

print('Callback ready.')


## Stage 1: Differential Evolution global search

Here Stage 1 runs differential evolution on the regularised misfit using strategy `randtobest1bin`, population size 20, and warm‑start initialisation.
The cell sets up the population, launches DE for a fixed number of iterations, and reports total runtime, convergence status, and the best misfit found.

In [ ]:
# Stage 1: Differential Evolution global search
print("-" * 70)
print(f"STAGE 1 Differential Evolution  {nx}x{ny}x{nz} grid")
print(f"control pts: {n_ctrl_x}x{n_ctrl_y}={n_ctrl_x*n_ctrl_y}  "
      f"lambda_s={lambda_s}  lambda_d={lambda_d}")
print("-" * 70)

_hist_x.clear()
_hist_f.clear()
_iter[0] = 0
_best[0] = np.inf
_t0[0]   = time.time()

popsize  = 15
n_params = n_ctrl_x * n_ctrl_y

# Warm-start: half the population around the depth prior, half from LHC.
# Noise sigma = depth_max * 0.40 (1200 m) so that DE mutation steps
# (~F * std ≈ 0.5 * 1200 = 600 m, 15% of [0,4000] range) are large
# enough to explore the landscape. The previous sigma=0.15 gave steps of
# only ~180 m (4.5% of range) — too tight for the weaker VD gravity signal.
rng = np.random.default_rng(42)
pop_prior = np.clip(
    _depth_ref[np.newaxis, :] +
    rng.normal(0, depth_max * 0.40, (popsize * n_params // 2, n_params)),
    depth_min_inv, depth_max_inv)
from scipy.stats import qmc
sampler  = qmc.LatinHypercube(d=n_params, seed=42)
pop_lhc  = qmc.scale(sampler.random(popsize * n_params - len(pop_prior)),
                     depth_min_inv, depth_max_inv)
init_pop = np.vstack([pop_prior, pop_lhc])

de = differential_evolution(
    misfit,
    bounds=bounds,
    strategy="randtobest1bin",
    maxiter=600,
    popsize=popsize,
    tol=1e-9,
    mutation=(0.5, 1.0),    # raised lower bound: 0.4→0.5 for better exploration
    recombination=0.85,     # slightly lower: 0.90→0.85 increases trial diversity
    polish=False,
    seed=42,
    disp=False,
    callback=de_callback,
    init=init_pop,
    workers=1,
)

print(f"\nDE finished in {time.time()-_t0[0]:.1f}s")
print(f"converged  = {de.success}")
print(f"best misfit = {de.fun:.8f}")


## Stage 2: Regularised L-BFGS-B local polish

Starting from the DE result, this cell uses L‑BFGS‑B to refine the solution while keeping the regularised misfit and depth bounds.  
It prints the runtime, convergence flag, DE vs L‑BFGS‑B misfits, and the improvement, showing how much local optimisation sharpens the geometry.

In [ ]:
# Stage 2: L-BFGS-B local polish (regularised)
print('=' * 65)
print('  STAGE 2 : L-BFGS-B  (regularised polish)')
print('=' * 65)
t_lb = time.time()

lb_result = minimize(
    misfit, x0=de.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-15, 'gtol': 1e-10, 'disp': True},
)

print(f'\nStage 2 finished in {time.time()-t_lb:.1f}s')
print(f'  converged  : {lb_result.success}')
print(f'  DE  misfit : {de.fun:.8f}')
print(f'  LB  misfit : {lb_result.fun:.8f}')
print(f'  Improvement: {de.fun - lb_result.fun:.8f}')


## Stage 3a and 3b: Data-only L-BFGS-B polishes

Two further L‑BFGS‑B runs minimise the data‑only misfit: Stage 3a with standard tolerances and Stage 3b with ultra‑tight tolerances.  
The cell compares data misfits from the Stage 2, 3a, and 3b solutions, selects the best parameter set, and reports the final data misfit.

In [ ]:
# Stage 3: Final regularised L-BFGS-B polish with ultra-tight tolerances
# FIX 7: Replaced data-only polishing with regularised polish.
#        Data-only misfit (Stage 3 original) drops the depth prior which
#        was the only term constraining depth uniqueness. Without it the
#        optimizer fits observational noise at the cost of depth accuracy —
#        RMS gravity residual falls but RMS depth error rises. This is the
#        classic underdetermined inversion trade-off.
# FIX 8: Model selection uses the regularised (full) misfit, not data-only.
print("=" * 65)
print("  STAGE 3 : Ultra-tight regularised L-BFGS-B")
print("=" * 65)
t_lb2 = time.time()
lb2_result = minimize(
    misfit, x0=lb_result.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-16, 'gtol': 1e-11, 'disp': True},
)
print(f'Stage-3 finished in {time.time()-t_lb2:.1f}s  misfit={lb2_result.fun:.8f}')

# Select best by regularised misfit (the objective that encodes depth knowledge)
candidates = [(de.x,         misfit(de.x)),
              (lb_result.x,  lb_result.fun),
              (lb2_result.x, lb2_result.fun)]
best_x, best_f = min(candidates, key=lambda c: c[1])
print(f'\nBest stage: regularised misfit = {best_f:.8f}')


## Recovered basin surface and residuals

Using the best parameter vector, this cell reconstructs the recovered basin surface and recomputes its gravity anomaly with variable density. 
It forms residuals, computes RMS gravity and RMS depth errors, and prints true vs recovered maximum depth to quantify inversion accuracy.

In [ ]:
# Recovered surface and residuals
recovered = surface_from_params(best_x, xc, yc)

gz_rec   = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_density(recovered))
residual = gz_rec - gz_obs
rms_grav  = float(np.sqrt(np.mean(residual**2)))
rms_depth = float(np.sqrt(np.mean((recovered - Z_basin_true)**2)))

print(f'RMS gravity residual : {rms_grav:.4f} mGal')
print(f'RMS depth error      : {rms_depth:.1f} m')
print(f'True depth max       : {Z_basin_true.max():.1f} m')
print(f'Recovered depth max  : {recovered.max():.1f} m')


## Coordinate arrays and cross-section location

This plotting-setup cell converts x and y to kilometres and builds meshgrids for map plots, then defines a cross-section at \(x = 7.5\) km.
It prints the chosen cross-section location, preparing for subsequent depth and residual cross-section visualisations.

In [ ]:
# ── Coordinate arrays for plotting ─────────────────────────────────────────
xkm = xc / 1000.0
ykm = yc / 1000.0
XX_mod, YY_mod = np.meshgrid(xkm, ykm, indexing='ij')

xi_cross = 7_500.0
xi_label = xi_cross / 1000.0
print(f'Cross-section at x = {xi_label:.1f} km')

## Plot (a): Gravity anomaly maps

In [ ]:
# ── Plot (a): Gravity anomaly maps ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle(
    '(a) Gravity Anomaly Maps — Model 3 (Vertically Variable Density)',
    fontsize=13, fontweight='bold',
)

gz_min = float(min(gz_obs.min(), gz_rec.min()))
gz_max = float(max(gz_obs.max(), gz_rec.max()))
vlim_r = float(np.max(np.abs([residual.min(), residual.max()])))

datasets = [gz_obs,   gz_rec,   residual]
titles   = ['Observed Anomaly (mGal)', 'Recovered Anomaly (mGal)', 'Residual Anomaly (mGal)']
cmaps    = ['jet',    'jet',    'RdBu_r']
vlims    = [(gz_min, gz_max), (gz_min, gz_max), (-vlim_r, vlim_r)]

for ax, dat, ttl, cmp, (vlo, vhi) in zip(axes, datasets, titles, cmaps, vlims):
    lv = np.linspace(vlo, vhi, 200)
    cf = ax.contourf(XX_mod, YY_mod, dat, levels=lv, cmap=cmp, extend='both')
    ax.set_title(ttl, fontweight='bold', fontsize=11)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11)
    ax.set_xlim(xkm[0], xkm[-1])
    ax.set_ylim(ykm[0], ykm[-1])
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    ax.axvline(xi_label, color='white', linestyle='--', linewidth=1.4,
               label=f'x={xi_label:.1f} km')
    cb = plt.colorbar(cf, ax=ax, pad=0.02)
    cb.locator   = mticker.MaxNLocator(nbins=6)
    cb.formatter = mticker.FormatStrFormatter('%.0f')
    cb.update_ticks()
    cb.set_label('gz (mGal)' if 'Residual' not in ttl else 'Residual gz (mGal)',
                 fontsize=10)

plt.tight_layout()
plt.savefig('model3_vd_a_gravity.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model3_vd_a_gravity.png')

## Mean density contrast of true and recovered basins
The vertically averaged density contrast is computed for both true and recovered surfaces to compare effective density structures under the variable density law.

In [ ]:
# ── Mean density contrast maps ──────────────────────────────────────────────
def mean_density_contrast(depth_surface):
    dz_val = ze[1] - ze[0]
    nx_l, ny_l = depth_surface.shape
    num = np.zeros((nx_l, ny_l))
    for k in range(nz):
        drho_k = density_contrast_at_depth(zc[k])
        dz_k   = np.minimum(dz_val,
                            np.maximum(0, depth_surface - (zc[k] - dz_val / 2)))
        num   += drho_k * dz_k
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(depth_surface > 0, num / depth_surface, 0.0)

drho_true_mean = mean_density_contrast(Z_basin_true)
drho_rec_mean  = mean_density_contrast(recovered)

print('Mean density contrast maps computed.')

## Plot (b): Basin depth & density structure

In [ ]:
# ── Plot (b): Basin depth & density structure ───────────────────────────────
vmax_d   = float(max(Z_basin_true.max(), recovered.max()))
levels_d = np.linspace(0, vmax_d, 60)

vmin_r   = min(drho_true_mean.min(), drho_rec_mean.min())
vmax_r   = max(drho_true_mean.max(), drho_rec_mean.max())
levels_r = np.linspace(vmin_r, vmax_r, 60)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    '(b) Basin Depth & Density Structure — Model 3 (Vertically Variable Density)',
    fontsize=13, fontweight='bold',
)

last_depth_cf = None
for ax, (dat, ttl) in zip(axes[0],
        [(Z_basin_true, 'True Basin Depth (m)'),
         (recovered,    'Recovered Basin Depth (m)')]):
    cf = ax.contourf(XX_mod, YY_mod, dat, levels=levels_d, cmap='jet', extend='max')
    ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11, labelpad=10)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11, labelpad=8)
    ax.set_xlim(xkm[0], xkm[-1])
    ax.set_ylim(ykm[0], ykm[-1])
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.tick_params(axis='x', pad=6)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)
    ax.axvline(xi_label, color='white', linestyle='--', linewidth=1.6)
    last_depth_cf = cf

last_rho_cf = None
for ax, (ttl, dat) in zip(axes[1],
        [('True Mean Density Contrast (kg/m3)',      drho_true_mean),
         ('Recovered Mean Density Contrast (kg/m3)', drho_rec_mean)]):
    cf = ax.contourf(XX_mod, YY_mod, dat, levels=levels_r, cmap='seismic_r', extend='both')
    ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11, labelpad=10)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11, labelpad=8)
    ax.set_xlim(xkm[0], xkm[-1])
    ax.set_ylim(ykm[0], ykm[-1])
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.tick_params(axis='x', pad=6)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)
    ax.axvline(xi_label, color='black', linestyle='--', linewidth=1.4)
    last_rho_cf = cf

fig.subplots_adjust(hspace=0.45, wspace=0.3, right=0.88)

cax_d = fig.add_axes([0.91, 0.54, 0.018, 0.34])
cb_d  = fig.colorbar(last_depth_cf, cax=cax_d, orientation='vertical')
cb_d.set_label('Depth (m)', fontweight='bold', fontsize=11, labelpad=12)
cb_d.ax.tick_params(labelsize=9)

cax_r = fig.add_axes([0.91, 0.10, 0.018, 0.34])
cb_r  = fig.colorbar(last_rho_cf, cax=cax_r, orientation='vertical')
cb_r.set_label('Mean dr (kg/m3)', fontweight='bold', fontsize=11, labelpad=12)
cb_r.ax.tick_params(labelsize=9)

plt.savefig('model3_vd_b_depth_density.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model3_vd_b_depth_density.png')

## Plot (c): Vertical cross-section at x = 7.5 km

In [ ]:
# ── Plot (c): Vertical cross-section at x = 7.5 km ─────────────────────────
ix_mod     = int(np.argmin(np.abs(xc - xi_cross)))
iy_mid_mod = ny // 2

true_depth_slice = Z_basin_true[ix_mod, :]
rec_depth_slice  = recovered[ix_mod, :]

drho_z_true = np.array([density_contrast_at_depth(zk)
                         if zk < Z_basin_true[ix_mod, iy_mid_mod] else 0.0
                         for zk in zc])
drho_z_rec  = np.array([density_contrast_at_depth(zk)
                         if zk < recovered[ix_mod, iy_mid_mod] else 0.0
                         for zk in zc])

z_line   = np.linspace(0, depth_max, 200)
drho_law = drho0 + alpha * z_line

z_fine      = np.linspace(0, Lz, 300)
Yg, Zg      = np.meshgrid(ykm, z_fine, indexing='ij')
inside_mask = Zg < rec_depth_slice[:, np.newaxis]
drho_grid   = drho0 + alpha * z_fine
drho_2d     = np.where(inside_mask, drho_grid[np.newaxis, :], np.nan)
vmin_d2     = np.nanmin(drho_2d)
vmax_d2     = np.nanmax(drho_2d)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5))
fig.suptitle(
    f'(c) Vertical Cross-Section at x = {xi_label:.1f} km — Model 3 (Vertically Variable Density)',
    fontsize=13, fontweight='bold',
)

pm = ax1.pcolormesh(
    ykm, z_fine, drho_2d.T,
    cmap='coolwarm_r', vmin=vmin_d2, vmax=vmax_d2,
    shading='auto', zorder=1)

ax1.plot(ykm, rec_depth_slice,
         color='red', linewidth=2.7, label='Recovered Basin', zorder=3)
ax1.plot(ykm, true_depth_slice,
         linestyle='--', color='blue', linewidth=2.2, label='True Basin', zorder=4)
ax1.fill_between(ykm, true_depth_slice, rec_depth_slice,
                 where=(rec_depth_slice > true_depth_slice),
                 alpha=0.25, color='orange', label='Over-estimated', zorder=2)
ax1.fill_between(ykm, true_depth_slice, rec_depth_slice,
                 where=(rec_depth_slice < true_depth_slice),
                 alpha=0.25, color='green', label='Under-estimated', zorder=2)

cb1 = plt.colorbar(pm, ax=ax1, pad=0.02, fraction=0.046)
cb1.set_label('dr inside recovered basin (kg/m3)', fontweight='bold', fontsize=10)
cb1.ax.tick_params(labelsize=9)

ax1.set_title(f'Basin Geometry along Y  (x = {xi_label:.1f} km)',
              fontweight='bold', fontsize=12)
ax1.set_xlabel('y (km)', fontweight='bold', fontsize=12)
ax1.set_ylabel('Depth (m)', fontweight='bold', fontsize=12)
ax1.set_ylim(0, Lz)
ax1.invert_yaxis()
ax1.xaxis.set_major_locator(MultipleLocator(5))
ax1.yaxis.set_major_locator(MultipleLocator(500))
for lbl in ax1.get_xticklabels() + ax1.get_yticklabels():
    lbl.set_fontweight('bold')
ax1.tick_params(labelsize=11)
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)
ax1.legend(prop={'weight': 'bold', 'size': 10}, loc='lower center',
           ncol=2, framealpha=0.85)
ax1.text(0.02, 0.04,
         f'RMS gravity: {rms_grav:.3f} mGal\nRMS depth:   {rms_depth:.1f} m',
         transform=ax1.transAxes, fontsize=10, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85))

ax2.fill_betweenx(zc / 1000.0, drho_z_rec,  0,
                  alpha=0.28, color='red',  label='Recovered fill', step='mid')
ax2.fill_betweenx(zc / 1000.0, drho_z_true, 0,
                  alpha=0.18, color='blue', label='True fill',      step='mid')
ax2.step(drho_z_true, zc / 1000.0,
         where='mid', linestyle='--', color='blue', linewidth=2.2, label='True dr(z)')
ax2.step(drho_z_rec,  zc / 1000.0,
         where='mid', color='red',  linewidth=2.7, label='Recovered dr(z)')
ax2.plot(drho_law, z_line / 1000.0,
         color='gray', linewidth=1.5, linestyle=':', label='dr law (full depth)')
ax2.axvline(0, color='black', linewidth=0.8, linestyle='-')

ax2.set_title(
    f'Density-Depth Profile at Basin Centre\n'
    f'(x={xi_label:.1f} km,  y={ykm[iy_mid_mod]:.1f} km)',
    fontweight='bold', fontsize=12)
ax2.set_xlabel('Density Contrast dr (kg/m3)', fontweight='bold', fontsize=12)
ax2.set_ylabel('Depth (km)', fontweight='bold', fontsize=12)
ax2.invert_yaxis()
ax2.yaxis.set_major_locator(MultipleLocator(0.5))
for lbl in ax2.get_xticklabels() + ax2.get_yticklabels():
    lbl.set_fontweight('bold')
ax2.tick_params(labelsize=11)
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax2.legend(prop={'weight': 'bold', 'size': 10}, loc='lower right')

plt.tight_layout()
plt.savefig('model3_vd_c_xsection.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model3_vd_c_xsection.png')

In [ ]:
# Final summary
print("=" * 65)
print("     INVERSION SUMMARY -- Model 3 (SINGLE BASIN, VERTICALLY VARIABLE DENSITY)")
print("=" * 65)
print(f'  Density law          : drho(z) = {drho0} + {alpha}*z  kg/m3')
print(f'  Grid                 : {nx}x{ny}x{nz}  (full resolution)')
print(f'  Control pts          : {n_ctrl_x}x{n_ctrl_y} = {n_ctrl_x*n_ctrl_y}')
print(f'  Regularisation       : lambda_s={lambda_s}  lambda_d={lambda_d}')
print(f'  Stage 1 (DE)         : strategy=randtobest1bin  popsize={popsize}  maxiter=2000')
print(f'    DE converged       : {de.success}')
print(f'    DE misfit          : {de.fun:.8f}')
print(f'  Stage 2 (L-BFGS-B)  : ftol=1e-15  gtol=1e-10  maxiter=5000')
print(f'    LB converged       : {lb_result.success}')
print(f'    LB misfit          : {lb_result.fun:.8f}')
print(f'  Stage 3 (ultra-tight): ftol=1e-16  gtol=1e-11  maxiter=5000')
print(f'    LB2 converged      : {lb2_result.success}')
print(f'    LB2 misfit         : {lb2_result.fun:.8f}')
print(f'  Best misfit          : {best_f:.8f}')
print(f'  RMS gravity residual : {rms_grav:.4f} mGal')
print(f'  RMS depth error      : {rms_depth:.1f} m')
print(f'  True depth max       : {Z_basin_true.max():.1f} m')
print(f'  Recovered depth max  : {recovered.max():.1f} m')
print("=" * 65)


## DE Convergence Curve

In [ ]:

# ── Plot (d): DE convergence curve ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(range(1, len(_hist_f) + 1), _hist_f,
            '-o', markersize=3, color='steelblue', linewidth=1.8)
ax.set_title('DE Convergence Curve — Model 3 (Single Basin - Variable Density)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('DE Iteration', fontweight='bold', fontsize=12)
ax.set_ylabel('Best Misfit (normalised MSE)', fontweight='bold', fontsize=12)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
for l in ax.get_xticklabels() + ax.get_yticklabels():
    l.set_fontweight('bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('model3_de_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model3_de_convergence.png')

## Cost-function topography in PCA space

In [ ]:
# ── 1. Build ensemble of "acceptable" models and their misfits ───────────────
misfit_array = np.array(_hist_f)
models_array = np.array(_hist_x)          # shape (N_models, n_params)
models_array = models_array.T             # shape (n_params, N_models)

misfit_threshold = np.percentile(misfit_array, 40.0)  # best 40% of models
mask_ok = misfit_array <= misfit_threshold

cost_finall  = misfit_array[mask_ok]     # (N_ok,)
model_finall = models_array[:, mask_ok]  # (n_params, N_ok)

print(f"Accepted models for PCA: {model_finall.shape[1]}")

# ── 2. PCA reduction ──────────────────────────────────────────────────────────
def pca_reduction_py(data):
    mean_vec = np.mean(data, axis=1, keepdims=True)
    data_z   = data - mean_vec
    C = np.cov(data_z)
    Evals, W_col = np.linalg.eigh(C)
    idx     = np.argsort(Evals)[::-1]
    Evalues = Evals[idx]
    W       = W_col[:, idx].T   # rows = eigenvectors
    pc      = W @ data_z
    return pc, Evalues, W, mean_vec

pc, Evalues, W, mean_model = pca_reduction_py(model_finall)

# ── 3. Cost-function topography in the PC1-PC2 plane ─────────────────────────
x = pc[0, :]   # PC1 scores
y = pc[1, :]   # PC2 scores

nxg, nyg = 80, 80
xg = np.linspace(x.min(), x.max(), nxg)
yg = np.linspace(y.min(), y.max(), nyg)
Xg, Yg = np.meshgrid(xg, yg, indexing="ij")

Vq = griddata(points=np.vstack([x, y]).T,
              values=cost_finall,
              xi=(Xg, Yg),
              method="linear")

plt.figure(figsize=(7, 5))
cs   = plt.contourf(Xg, Yg, Vq, levels=12, cmap="jet")
cbar = plt.colorbar(cs)
cbar.set_label("Regularised misfit (dimensionless)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Cost-function topography in PCA space (single basin-variable density) noisy data")

# ── 4. Project best model and true model into PCA space ──────────────────────
# FIX: Use best_x (post all polishing), and Z_basin_true sampled on ctrl grid.
params_best = best_x

from scipy.interpolate import RegularGridInterpolator
_interp_true = RegularGridInterpolator(
    (xc, yc), Z_basin_true, method="linear",
    bounds_error=False, fill_value=0.0)
_X_ctrl_2d, _Y_ctrl_2d = np.meshgrid(x_ctrl, y_ctrl, indexing="ij")
true_on_ctrl = _interp_true(
    np.column_stack([_X_ctrl_2d.ravel(), _Y_ctrl_2d.ravel()])
).reshape(n_ctrl_x, n_ctrl_y)
true_model = true_on_ctrl.ravel()

mean_flat     = mean_model.ravel()
best_centered = params_best - mean_flat
true_centered = true_model  - mean_flat

loc_best = W @ best_centered
loc_true = W @ true_centered

plt.plot(loc_best[0], loc_best[1], "r^", markersize=10, label="Best model (post L-BFGS-B)")
plt.plot(loc_true[0], loc_true[1], "gv", markersize=10, label="True model")
plt.legend(loc="best")
plt.tight_layout()
plt.savefig('model3_pca_noisy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best model  PC1={loc_best[0]:.4f}  PC2={loc_best[1]:.4f}")
print(f"True model  PC1={loc_true[0]:.4f}  PC2={loc_true[1]:.4f}")
dist = np.sqrt((loc_best[0]-loc_true[0])**2 + (loc_best[1]-loc_true[1])**2)
print(f"PC-space distance: {dist:.4f}")
